# Laws → Qdrant — Notebook (using `configs.py`)

In [1]:
# Cell 1: Setup and imports
import sys
import os
from pathlib import Path

# Add parent directory to path
notebook_path = Path().resolve()
parent_path = notebook_path.parent
sys.path.append(str(parent_path))

from dotenv import load_dotenv
load_dotenv()

# Verify environment variables
CONGRESS_API_KEY = os.getenv("CONGRESS_API_KEY")
if not CONGRESS_API_KEY:
    raise ValueError("CONGRESS_API_KEY not found in .env")

print("✓ Environment loaded")

✓ Environment loaded


In [2]:
from src.backend.law_parser.congress_laws_parser import CongressAPIClient, CongressLawsQdrantProcessor, BillInfo

print("✓ Imports successful")

✓ Imports successful


In [3]:
# Cell 3: Initialize processor
processor = CongressLawsQdrantProcessor(CONGRESS_API_KEY)

print("✓ Processor initialized")
print(f"✓ Qdrant URL: {processor.qdrant_manager.configs.vectordb.URL}")

2025-10-06 12:41:25,035 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2025-10-06 12:41:25,550 - INFO - HTTP Request: GET http://localhost:6333/collections "HTTP/1.1 200 OK"


✓ Processor initialized
✓ Qdrant URL: http://localhost:6333


In [ ]:
# Cell 4: Fetch and filter 10 relevant laws
client = CongressAPIClient(CONGRESS_API_KEY)

all_bills = client.get_enacted_bills([118, 127], limit=100)
print(f"Found {len(all_bills)} enacted bills")

relevant_bills = []
for bill in all_bills[:50]:
    congress = bill.get('congress')
    bill_type = bill.get('type', '').lower()
    bill_number = bill.get('number')
    title = bill.get('title', '')
    
    if not all([congress, bill_type, bill_number]):
        continue
    
    subjects = client.get_bill_subjects(congress, bill_type, bill_number)
    
    if processor.is_relevant_law(subjects, title):
        relevant_bills.append(bill)
        print(f"✓ {len(relevant_bills)}. {title[:80]}...")
        
        if len(relevant_bills) >= 10:
            break

print(f"\nFound {len(relevant_bills)} relevant laws")
print(relevant_bills)

2025-10-06 12:41:33,118 - INFO - Fetching bills from 118th Congress...
2025-10-06 12:41:35,395 - INFO - Found 10 enacted hr bills in 118th Congress
2025-10-06 12:41:36,163 - INFO - Found 3 enacted s bills in 118th Congress
2025-10-06 12:41:37,517 - INFO - Fetching bills from 127th Congress...
2025-10-06 12:41:39,657 - INFO - Total enacted bills found: 13


Found 13 enacted bills
✓ 1. Social Security Fairness Act of 2023...
✓ 2. Thomas R. Carper Water Resources Development Act of 2024...
✓ 3. Senator Elizabeth Dole 21st Century Veterans Healthcare and Benefits Improvement...

Found 3 relevant laws
[{'congress': 118, 'latestAction': {'actionDate': '2025-01-05', 'text': 'Became Public Law No: 118-273.'}, 'number': '82', 'originChamber': 'House', 'originChamberCode': 'H', 'title': 'Social Security Fairness Act of 2023', 'type': 'HR', 'updateDate': '2025-05-27', 'updateDateIncludingText': '2025-05-27', 'url': 'https://api.congress.gov/v3/bill/118/hr/82?format=json'}, {'congress': 118, 'latestAction': {'actionDate': '2025-01-04', 'text': 'Became Public Law No: 118-272.'}, 'number': '4367', 'originChamber': 'Senate', 'originChamberCode': 'S', 'title': 'Thomas R. Carper Water Resources Development Act of 2024', 'type': 'S', 'updateDate': '2025-05-27', 'updateDateIncludingText': '2025-05-27', 'url': 'https://api.congress.gov/v3/bill/118/s/4367?fo

In [5]:
 print(f"\nProcessing: {title[:60]}...")


Processing: FISHES Act...


In [8]:
print(xml_text)
print(type(xml_text))

HR 5103 ENR: Fishery Improvement to Streamline untimely regulatory Hurdles post Emergency Situation Act U.S. House of Representatives text/xml EN Pursuant to Title 17 Section 105 of the United States Code, this file is not subject to copyright protection and is in the public domain. IB One Hundred Eighteenth Congress of the United States of America At the Second Session Begun and held at the City of Washington on Wednesday, the third day of January, two thousand and twenty-four H. R. 5103 AN ACT To require the Director of the Office of Management and Budget to approve or deny spend plans within a certain amount of time, and for other purposes. 1. Short title This Act may be cited as the Fishery Improvement to Streamline untimely regulatory Hurdles post Emergency Situation Act or the FISHES Act . 2. Spend plans Section 312(a)(6) of the Magnuson-Stevens Fishery Conservation and Management Act ( 16 U.S.C. 1861a(a)(6) ) is amended— (1) in subparagraph (D), to read as follows: (D) Spend pla

In [5]:
# Cell 5 (Fixed): Process and store laws in Qdrant
from src.backend.vector_db.chunker import create_chunks

stored_count = 0

for bill in relevant_bills:
    try:
        congress = bill.get('congress')
        bill_type = bill.get('type', '').lower()
        bill_number = bill.get('number')
        title = bill.get('title', '')
        
        print(f"\nProcessing: {title[:60]}...")
        
        xml_url = client.get_bill_xml_url(congress, bill_type, bill_number)
        if not xml_url:
            print("  ✗ No XML found")
            continue
        
        xml_text = client.download_and_clean_xml(xml_url)
        if not xml_text:
            print("  ✗ Failed to download")
            continue
        
        # Use the function directly instead of method
        chunks = create_chunks(xml_text)
        if not chunks:
            print("  ✗ No chunks created")
            continue
            
        latest_action = bill.get('latestAction', {})
        law_number = processor._extract_law_number(latest_action.get('text', ''))
        
        bill_info = BillInfo(
            congress=congress,
            bill_type=bill_type,
            bill_number=bill_number,
            title=title,
            enacted_date=latest_action.get('actionDate'),
            law_number=law_number
        )
        
        law_id = processor._create_law_id(bill_info)
        
        chunks_count = processor.qdrant_manager.process_and_store_law(xml_text, law_id)
        print(f"  ✓ Stored {chunks_count} chunks with ID: {law_id}")
        stored_count += 1
        
    except Exception as e:
        print(f"  ✗ Error: {e}")
        import traceback
        traceback.print_exc()
        continue

print(f"\n{'='*60}")
print(f"COMPLETED! Stored {stored_count} laws in Qdrant")
print(f"{'='*60}")


Processing: Social Security Fairness Act of 2023...


2025-10-06 12:42:00,673 - INFO - Found XML URL for HR.82: Enrolled Bill
2025-10-06 12:42:00,675 - INFO - Downloading XML from: https://www.congress.gov/118/bills/hr82/BILLS-118hr82enr.xml
2025-10-06 12:42:01,427 - WARNING - BeautifulSoup parsing failed: Couldn't find a tree builder with the features you requested: xml. Do you need to install a parser library?, trying basic cleaning
2025-10-06 12:42:03,898 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-10-06 12:42:03,961 - INFO - HTTP Request: PUT http://localhost:6333/collections/laws/points?wait=true "HTTP/1.1 200 OK"


  ✓ Stored 1 chunks with ID: congress_118_hr_82_pl_118_273

Processing: Thomas R. Carper Water Resources Development Act of 2024...


2025-10-06 12:42:04,545 - INFO - Found XML URL for S.4367: Enrolled Bill
2025-10-06 12:42:04,546 - INFO - Downloading XML from: https://www.congress.gov/118/bills/s4367/BILLS-118s4367enr.xml
2025-10-06 12:42:05,553 - WARNING - BeautifulSoup parsing failed: Couldn't find a tree builder with the features you requested: xml. Do you need to install a parser library?, trying basic cleaning
2025-10-06 12:42:06,872 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 400 Bad Request"


Error vectorizing chunk 1 for congress_118_s_4367_pl_118_272: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens, however you requested 138221 tokens (138221 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code': None}}
  ✓ Stored 0 chunks with ID: congress_118_s_4367_pl_118_272

Processing: Senator Elizabeth Dole 21st Century Veterans Healthcare and ...


2025-10-06 12:42:07,442 - INFO - Found XML URL for S.141: Enrolled Bill
2025-10-06 12:42:07,444 - INFO - Downloading XML from: https://www.congress.gov/118/bills/s141/BILLS-118s141enr.xml
2025-10-06 12:42:08,306 - WARNING - BeautifulSoup parsing failed: Couldn't find a tree builder with the features you requested: xml. Do you need to install a parser library?, trying basic cleaning
2025-10-06 12:42:09,053 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 400 Bad Request"


Error vectorizing chunk 1 for congress_118_s_141_pl_118_210: Error code: 400 - {'error': {'message': "This model's maximum context length is 8192 tokens, however you requested 51356 tokens (51356 in your prompt; 0 for the completion). Please reduce your prompt; or completion length.", 'type': 'invalid_request_error', 'param': None, 'code': None}}
  ✓ Stored 0 chunks with ID: congress_118_s_141_pl_118_210

COMPLETED! Stored 3 laws in Qdrant


In [7]:
collection_info = processor.qdrant_manager.client.get_collection("laws")
print(f"Collection: {collection_info.status}")
print(f"Total points: {collection_info.points_count}")

2025-10-02 19:33:26,243 - INFO - HTTP Request: GET http://localhost:6333/collections/laws "HTTP/1.1 200 OK"


Collection: green
Total points: 8
